In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn  as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, random_split
from torch.utils.data import Subset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

Training device: cuda


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
train_data = torchvision.datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform
)


train_loader = torch.utils.data.DataLoader(train_data, batch_size= 32, shuffle=True,num_workers=2)
test_loader =torch.utils.data.DataLoader(test_data, batch_size = 32, shuffle =True, num_workers=2)

8.3%

In [ ]:
generator = torch.Generator().manual_seed(42)

train_subset, val_subset = random_split(
    train_data,
    [45000, 5000],
    generator=generator
)

train_loader = DataLoader(
    train_subset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_subset, batch_size=128, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_data, batch_size=128, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

In [ ]:
image , label  = train_data [0]

In [ ]:
image.size()

In [ ]:
class_names = ['plane','car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship','truck']

In [ ]:
class NeuralNet(nn.Module):

  def __init__(self):
    super(NeuralNet, self).__init__()

    self.conv1 =nn.Conv2d(3,12,5) ##32 - 5 = 27 / 1   +1 so finally (12,28,28)
    self.pool = nn.MaxPool2d(2,2)
    self.conv2 = nn.Conv2d(12,24,5)
    self.fc1 = nn.Linear(24 *5 * 5,120)
    self.fc2 = nn.Linear(120,84)
    self.fc3 = nn.Linear(84,10)

  def forward(self,x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [ ]:
net = NeuralNet().to(device)

loss_function = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    net.parameters(),
    lr=0.001,
    momentum=0.9
)

print("Model device:", next(net.parameters()).device)
print(
    "Trainable parameters:",
    sum(p.numel() for p in net.parameters() if p.requires_grad)
)

In [ ]:
import time

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
EPOCHS = 50

for epoch in range(EPOCHS):

    start_time = time.time()

    # TRAINING
    net.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = net(images)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predicted = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc = 100 * correct / total

    # VALIDATION
    net.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = net(images)
            loss = loss_function(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            predicted = outputs.argmax(dim=1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / val_total
    val_acc = 100 * val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(net.state_dict(), "best_model.pth")

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}% | "
        f"Time: {elapsed:.1f}s"
    )

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(14, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Training Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], label="Training Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=300)
plt.show()

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

net.load_state_dict(
    torch.load("best_model.pth", map_location=device, weights_only=True)
)

net.to(device)
net.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)

        outputs = net(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Test Accuracy:", accuracy_score(
    all_labels, all_predictions
) * 100)

print("\nClassification Report:\n")

print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4
))

In [ ]:
import seaborn as sns

cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("CIFAR-10 Confusion Matrix")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

In [ ]:
for i, class_name in enumerate(class_names):

    class_total = cm[i].sum()
    class_correct = cm[i, i]

    accuracy = 100 * class_correct / class_total

    print(f"{class_name:10s}: {accuracy:.2f}%")

In [ ]:
import random

net.eval()

images, labels = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    outputs = net(images)
    probabilities = torch.softmax(outputs, dim=1)
    confidences, predictions = probabilities.max(dim=1)

images = images.cpu()
labels = labels.cpu()

plt.figure(figsize=(14, 8))

for i in range(12):

    plt.subplot(3, 4, i + 1)

    image = images[i] * 0.5 + 0.5
    image = image.permute(1, 2, 0)

    true_label = class_names[labels[i]]
    predicted_label = class_names[predictions[i].item()]
    confidence = confidences[i].item() * 100

    plt.imshow(image)
    plt.title(
        f"True: {true_label}\nPred: {predicted_label} ({confidence:.1f}%)"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()